# ILSA Survey Analysis - Example Notebook

This notebook demonstrates how to work with the ILSA survey articles dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.analysis.corpus import build_frames, load_corpus

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load the corpus

`load_corpus` reads `ilsa_survey_articles/json` and collapses duplicate extractions of the same article (219 files → 212 articles).

In [ ]:
articles, warnings = load_corpus()
frames = build_frames(articles)
print(f"Loaded {len(articles)} articles")

## 2. Convert to DataFrame

In [ ]:
# Create DataFrame
df = pd.json_normalize(articles)
print(f"DataFrame shape: {df.shape}")
print("\nColumns:")
print(df.columns.tolist()[:20])

## 3. Basic Statistics

In [ ]:
# Publication years
year_counts = df['metadata.year'].value_counts().sort_index()
plt.figure(figsize=(10, 5))
year_counts.plot(kind='bar')
plt.title('Publication Years')
plt.xlabel('Year')
plt.ylabel('Number of Articles')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. ML Techniques Analysis

In [ ]:
# Canonical model names (SVM = Support Vector Machine = SVR, …), one count per article,
# predictive learners only (SHAP / TF-IDF / clustering excluded)
am = frames['article_models']
technique_counts = am.groupby('model')['article_id'].nunique().sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 6))
technique_counts.plot(kind='barh')
plt.title('Most Common ML Models (articles)')
plt.xlabel('Number of articles')
plt.tight_layout()
plt.show()

## 5. Variable Categories

In [ ]:
# Extract variable categories
categories = []
for article in articles:
    for confounder in article['data']['confounders_identified']:
        categories.append(confounder['category'])

category_counts = pd.Series(categories).value_counts()

plt.figure(figsize=(10, 6))
category_counts.plot(kind='pie', autopct='%1.1f%%')
plt.title('Variable Categories Distribution')
plt.ylabel('')
plt.tight_layout()
plt.show()

## 6. RQ outputs

The RQ1 literature tables are written by `scripts/rq1_literature_analysis.py` to `outputs/rq1/`; the knowledge base by `scripts/build_predictor_knowledge_base.py` to `outputs/knowledge_base/`.

In [ ]:
freq = pd.read_csv('outputs/rq1/model_frequency.csv')
freq[freq['subset'] == 'achievement_ml'].head(10)